# 03 · Silver: clean and validated tables
Splits violations and enforcement actions into separate tables (one row per entity),
converts data types and gives columns readable names.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_silver.violations AS
SELECT
  PWSID                                                              AS pwsid,
  VIOLATION_ID                                                       AS violation_id,
  VIOLATION_CODE                                                     AS violation_code,
  VIOLATION_CATEGORY_CODE                                            AS violation_category_code,
  IS_HEALTH_BASED_IND = 'Y'                                          AS is_health_based,
  IS_MAJOR_VIOL_IND = 'Y'                                            AS is_major_violation,
  CONTAMINANT_CODE                                                   AS contaminant_code,
  RULE_CODE                                                          AS rule_code,
  RULE_FAMILY_CODE                                                   AS rule_family_code,
  VIOLATION_STATUS                                                   AS violation_status,
  to_date(try_to_timestamp(NON_COMPL_PER_BEGIN_DATE, 'MM/dd/yyyy'))  AS noncompliance_begin_date,
  to_date(try_to_timestamp(NON_COMPL_PER_END_DATE, 'MM/dd/yyyy'))    AS noncompliance_end_date,
  to_date(try_to_timestamp(CALCULATED_RTC_DATE, 'MM/dd/yyyy'))       AS returned_to_compliance_date,
  _release
FROM workspace.sdwa_bronze.violations_enforcement
WHERE VIOLATION_ID IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY PWSID, VIOLATION_ID
  ORDER BY try_to_timestamp(VIOL_LAST_REPORTED_DATE, 'MM/dd/yyyy') DESC
) = 1

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_silver.enforcement_actions AS
SELECT DISTINCT
  PWSID                                                      AS pwsid,
  VIOLATION_ID                                               AS violation_id,
  ENFORCEMENT_ID                                             AS enforcement_id,
  to_date(try_to_timestamp(ENFORCEMENT_DATE, 'MM/dd/yyyy'))  AS enforcement_date,
  ENFORCEMENT_ACTION_TYPE_CODE                               AS enforcement_action_type_code,
  ENF_ACTION_CATEGORY                                        AS enforcement_action_category,
  _release
FROM workspace.sdwa_bronze.violations_enforcement
WHERE ENFORCEMENT_ID IS NOT NULL

In [0]:
%sql
SELECT
  COUNT(*)                                                AS violations,
  COUNT_IF(noncompliance_begin_date IS NULL)              AS missing_begin_date,
  COUNT_IF(is_health_based)                               AS health_based,
  (SELECT COUNT(*) FROM workspace.sdwa_silver.enforcement_actions) AS enforcement_actions
FROM workspace.sdwa_silver.violations

In [0]:
%sql
SELECT table_name, column_name, data_type
FROM workspace.information_schema.columns
WHERE table_schema = 'sdwa_bronze'
  AND table_name IN ('pub_water_systems', 'geographic_areas')
ORDER BY table_name, ordinal_position

In [0]:
%sql
SELECT table_name, column_name, data_type
FROM workspace.information_schema.columns
WHERE table_schema = 'sdwa_bronze'
  AND table_name IN ('pub_water_systems', 'geographic_areas')
ORDER BY table_name, ordinal_position

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_silver.water_systems AS
SELECT
  PWSID                                                   AS pwsid,
  PWS_NAME                                                AS system_name,
  PWS_ACTIVITY_CODE = 'A'                                 AS is_active,
  PWS_TYPE_CODE                                           AS system_type_code,
  CASE PWS_TYPE_CODE
    WHEN 'CWS'    THEN 'Community water system'
    WHEN 'NTNCWS' THEN 'Non-transient non-community'
    WHEN 'TNCWS'  THEN 'Transient non-community'
  END                                                     AS system_type,
  CASE OWNER_TYPE_CODE
    WHEN 'F' THEN 'Federal government'
    WHEN 'L' THEN 'Local government'
    WHEN 'M' THEN 'Public/private'
    WHEN 'N' THEN 'Native American'
    WHEN 'P' THEN 'Private'
    WHEN 'S' THEN 'State government'
  END                                                     AS owner_type,
  CASE GW_SW_CODE WHEN 'GW' THEN 'Groundwater' WHEN 'SW' THEN 'Surface water' END AS water_source,
  try_cast(POPULATION_SERVED_COUNT AS INT)                AS population_served,
  try_cast(SERVICE_CONNECTIONS_COUNT AS INT)              AS service_connections,
  IS_SCHOOL_OR_DAYCARE_IND = 'Y'                          AS is_school_or_daycare,
  CITY_NAME                                               AS city,
  ZIP_CODE                                                AS zip_code,
  to_date(try_to_timestamp(PWS_DEACTIVATION_DATE, 'MM/dd/yyyy')) AS deactivation_date,
  _release
FROM workspace.sdwa_bronze.pub_water_systems

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_silver.system_counties AS
SELECT DISTINCT
  PWSID                   AS pwsid,
  initcap(COUNTY_SERVED)  AS county,
  _release
FROM workspace.sdwa_bronze.geographic_areas
WHERE COUNTY_SERVED IS NOT NULL

In [0]:
%sql
SELECT
  COUNT(*)                                             AS systems,
  COUNT_IF(is_active)                                  AS active_systems,
  COUNT_IF(population_served IS NULL)                  AS missing_population,
  COUNT_IF(system_type IS NULL)                        AS unknown_type,
  (SELECT COUNT(DISTINCT pwsid) FROM workspace.sdwa_silver.system_counties) AS systems_with_county,
  (SELECT COUNT(*) FROM (SELECT pwsid FROM workspace.sdwa_silver.system_counties
                         GROUP BY pwsid HAVING COUNT(*) > 1))              AS multi_county_systems
FROM workspace.sdwa_silver.water_systems